In [1]:
import planetary_computer
import itertools
import dask.dataframe as dd

cc = planetary_computer.get_container_client("pcstacitems", "items")

blobs = list(cc.list_blobs("sentinel-2-l2a.parquet/"))

def key(blob):
    return blob.name.split("/")[1].split("_")[0]

keep_blobs = []
for k, v in itertools.groupby(sorted(blobs, key=key), key=key):
    v = list(v)
    blob = max(v, key=lambda x: x.last_modified)
    keep_blobs.append(blob)
    
uris = [f"az://items/{blob.name}" for blob in keep_blobs]

In [2]:
len(uris)

127

In [3]:
df = dd.read_parquet(uris, storage_options={"account_name": "pcstacitems", "credential": planetary_computer.sas.get_token("pcstacitems", "items").token})
df.head()

,assets,bbox,collection,geometry,id,links,stac_extensions,stac_version,type,constellation,...,s2:product_uri,s2:reflectance_conversion_factor,s2:saturated_defective_pixel_percentage,s2:snow_ice_percentage,s2:thin_cirrus_percentage,s2:unclassified_percentage,s2:vegetation_percentage,s2:water_percentage,sat:orbit_state,sat:relative_orbit
0,"{'AOT': {'gsd': 10.0, 'href': 'https://sentine...","{'xmin': 32.782321671748434, 'ymin': 71.805561...",sentinel-2-l2a,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1e\x00...",S2A_MSIL2A_20150704T101006_R022_T35XQA_2021041...,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/eo/v1.0.0/s...,1.0.0,Feature,Sentinel 2,...,S2A_MSIL2A_20150704T101006_N0212_R022_T35XQA_2...,0.967449,0.0,0.000000,0.497183,0.000000,0.000000,2.148608,descending,22
1,"{'AOT': {'gsd': 10.0, 'href': 'https://sentine...","{'xmin': 8.479220766509275, 'ymin': 41.4606964...",sentinel-2-l2a,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x00...,S2A_MSIL2A_20150704T101006_R022_T32TMM_2021041...,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/eo/v1.0.0/s...,1.0.0,Feature,Sentinel 2,...,S2A_MSIL2A_20150704T101006_N0212_R022_T32TMM_2...,0.967449,0.0,0.012059,0.077676,0.265107,59.675044,25.984192,descending,22
2,"{'AOT': {'gsd': 10.0, 'href': 'https://sentine...","{'xmin': -7.7299243, 'ymin': 45.0432611, 'xmax...",sentinel-2-l2a,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00 \x00\x0...,S2A_MSIL2A_20150715T112846_R037_T29TPL_2021041...,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/eo/v1.0.0/s...,1.0.0,Feature,Sentinel 2,...,S2A_MSIL2A_20150715T112846_N0212_R037_T29TPL_2...,0.967575,0.0,0.000000,0.000000,0.000000,0.000000,1.559118,descending,37
3,"{'AOT': {'gsd': 10.0, 'href': 'https://sentine...","{'xmin': 32.99946824095416, 'ymin': 69.6891048...",sentinel-2-l2a,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\t\x00\x...,S2A_MSIL2A_20150704T101006_R022_T36WWC_2021041...,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/eo/v1.0.0/s...,1.0.0,Feature,Sentinel 2,...,S2A_MSIL2A_20150704T101006_N0212_R022_T36WWC_2...,0.967449,0.0,0.000000,3.165583,0.323290,0.274893,11.357917,descending,22
4,"{'AOT': {'gsd': 10.0, 'href': 'https://sentine...","{'xmin': 4.999786508901344, 'ymin': 26.0978038...",sentinel-2-l2a,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,S2A_MSIL2A_20150704T101006_R022_T31RGK_2021041...,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/eo/v1.0.0/s...,1.0.0,Feature,Sentinel 2,...,S2A_MSIL2A_20150704T101006_N0212_R022_T31RGK_2...,0.967449,0.0,0.000000,6.375423,0.515758,0.000000,0.000007,descending,22


In [4]:
df.columns

Index(['assets', 'bbox', 'collection', 'geometry', 'id', 'links',
       'stac_extensions', 'stac_version', 'type', 'constellation', 'datetime',
       'eo:cloud_cover', 'instruments', 'platform', 'proj:epsg',
       's2:cloud_shadow_percentage', 's2:dark_features_percentage',
       's2:datastrip_id', 's2:datatake_id', 's2:datatake_type',
       's2:degraded_msi_data_percentage', 's2:generation_time',
       's2:granule_id', 's2:high_proba_clouds_percentage',
       's2:mean_solar_azimuth', 's2:mean_solar_zenith',
       's2:medium_proba_clouds_percentage', 's2:mgrs_tile',
       's2:nodata_pixel_percentage', 's2:not_vegetated_percentage',
       's2:processing_baseline', 's2:product_type', 's2:product_uri',
       's2:reflectance_conversion_factor',
       's2:saturated_defective_pixel_percentage', 's2:snow_ice_percentage',
       's2:thin_cirrus_percentage', 's2:unclassified_percentage',
       's2:vegetation_percentage', 's2:water_percentage', 'sat:orbit_state',
       'sat:relativ

In [5]:
print(len(df))

43266755


In [6]:
# df = df[['id', 'geometry', 'bbox', 'datetime', 'eo:cloud_cover', "s2:product_uri", "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]
df = df[['id', 'geometry', 'datetime', 'eo:cloud_cover', "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]

df["datetime"] = dd.to_datetime(df["datetime"])

In [7]:
df.head()

,id,geometry,datetime,eo:cloud_cover,s2:granule_id,s2:nodata_pixel_percentage,s2:saturated_defective_pixel_percentage
0,S2A_MSIL2A_20150704T101006_R022_T35XQA_2021041...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1e\x00...",2015-07-04 10:10:06.027000+00:00,97.851394,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133711_A0001...,20.722227,0.0
1,S2A_MSIL2A_20150704T101006_R022_T32TMM_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x00...,2015-07-04 10:10:06.027000+00:00,0.177088,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133213_A0001...,63.654864,0.0
2,S2A_MSIL2A_20150715T112846_R037_T29TPL_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00 \x00\x0...,2015-07-15 11:28:46.027000+00:00,98.440793,S2A_OPER_MSI_L2A_TL_ESRI_20210411T152630_A0003...,96.404278,0.0
3,S2A_MSIL2A_20150704T101006_R022_T36WWC_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\t\x00\x...,2015-07-04 10:10:06.027000+00:00,88.028410,S2A_OPER_MSI_L2A_TL_ESRI_20210411T133717_A0001...,88.475966,0.0
4,S2A_MSIL2A_20150704T101006_R022_T31RGK_2021041...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2015-07-04 10:10:06.027000+00:00,45.840056,S2A_OPER_MSI_L2A_TL_ESRI_20210411T132936_A0001...,0.000000,0.0


In [8]:
filtered_df = df[(df['datetime'] >= '2021-01-01') &
                #  (df['datetime'] < '2022-01-01') &
                 (df['eo:cloud_cover'] < 20) &
                 (df['s2:nodata_pixel_percentage'] < 10) &
                 (df['s2:saturated_defective_pixel_percentage'] < 10)]

In [9]:
filtered_df = filtered_df.compute()
print(len(filtered_df))

3962367


In [10]:
filtered_df.shape

(3962367, 7)

In [11]:
filtered_df.head()

,id,geometry,datetime,eo:cloud_cover,s2:granule_id,s2:nodata_pixel_percentage,s2:saturated_defective_pixel_percentage
26,S2B_MSIL2A_20210103T160649_R097_T16QGG_2021012...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...",2021-01-03 16:06:49.024000+00:00,0.996861,S2B_OPER_MSI_L2A_TL_ESRI_20210121T134229_A0200...,0.000000,0.0
27,S2B_MSIL2A_20210103T160649_R097_T16QGH_2021010...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x06\x00...,2021-01-03 16:06:49.024000+00:00,2.947462,S2B_OPER_MSI_L2A_TL_ESRI_20210105T044544_A0200...,0.000902,0.0
49,S2A_MSIL2A_20210127T001111_R073_T55KHP_2021012...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2021-01-27 00:11:11.024000+00:00,8.134838,S2A_OPER_MSI_L2A_TL_ESRI_20210127T105841_A0292...,0.000010,0.0
52,S2A_MSIL2A_20210103T033131_R018_T48PTV_2021010...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2021-01-03 03:31:31.024000+00:00,5.242336,S2A_OPER_MSI_L2A_TL_ESRI_20210103T184542_A0289...,0.000020,0.0
54,S2A_MSIL2A_20210103T222541_R029_T60HTF_2021010...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,2021-01-03 22:25:41.024000+00:00,10.478366,S2A_OPER_MSI_L2A_TL_ESRI_20210105T064210_A0289...,0.000003,0.0


In [12]:
max_date = filtered_df['datetime'].max().strftime("%Y_%m_%d")
print(max_date)

2026_01_12


In [14]:
filtered_df.to_parquet(f"s2l2a_clouds_lt_{max_date}_gt_2021_01_01.parquet", index=False)
print(f"Saved to: s2l2a_clouds_lt_{max_date}_gt_2021_01_01.parquet")

Saved to: s2l2a_clouds_lt_2026_01_12_gt_2021_01_01.parquet
